### Environment
+ Install **ollama**: `curl -fsSL https://ollama.com/install.sh | sh`
+ Pull a model to use with OpenAI API: `ollama pull gemma3:1b`
+ Install **Ollama Python**: `uv add ollama`

In [2]:
from ollama import chat

response = chat(
    model='gemma3:1b',
    messages=[{'role': 'user', 'content': 'Hello!'}],
)
print(response.message.content)

Hi there! 😊 How can I help you today? 😄



In [3]:
from openai import OpenAI

openai_client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

responses_result = openai_client.responses.create(
  model='gemma3:1b',
  input='Write a short poem about the color blue',
)
print(responses_result.output_text)

Okay, here’s a poem about Blue:

***

A canvas soft, serene and deep,
The tranquil hue that secrets keep.


Like glaciers shining in the light,
Blue washes over with its plight.




Rain hangs heavy, cool and free,
Lost in depths of a peaceful sea.



Sometimes misty fields ignite,
Burning hues across blue's delight. 

A moment calm, and truly grand,
Deep, endless, beautiful, Blue land.


### RAG

In [4]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gemma3:1b",
        input=prompt
    )
    return response.output_text

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Okay, let's figure that out for you! 😊 

To answer your question: **Yes, absolutely! You can definitely join the course right now.** 

**But...to help me give you a more complete and useful response, could you tell me which course it is?** I need to know what the course *is* so I can check its requirements or if there's anything preventing you from joining.


You can reply with:

*   The Course Name 
*   A link to the Course Page


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you want a certificate, you need to submit your project while we're still accepting submissions.


### Course FAQ dataset

In [9]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [11]:
documents[101]

{'id': '4277a9cb86',
 'course': 'data-engineering-zoomcamp',
 'section': 'Module 1: Postgres, pgAdmin & Python ingestion',
 'question': 'Postgres: "Column does not exist" but it actually does (Pyscopg2 error in MacBook Pro M2)',
 'answer': 'In join queries, if you mention the column name directly or enclose it in single quotes, you\'ll encounter an error saying "column does not exist".\n\n**Solution:** Enclose the column names in double quotes, and it will work correctly.'}

minsearch

In [13]:
# indexing
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [14]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [15]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [19]:
# Not all fields are equally important. The question field is usually more relevant than section for matching, hence the stronger signal
# Then filter by course name
def search(question, course="mlops-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [20]:
search_results = search(question)

In [21]:
[doc["question"] for doc in search_results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

### Build prompt

In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [23]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [26]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: Course - Can I still join the course after the start date?
A: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

General Course-Related Questions
Q: Homework: Just found this course, can I still submit homeworks?
A: To clarify on **late homework submissions**:

- You cannot submit after the homework is scored, as the form is closed.
- Once the form is closed (i.e., scored), no further submissions are possible.
- You can check your code against the solution by reviewing the `homework.md` file.

If the due date has passed but the form is still "Open/Submittable":

- This is considered a "late homework submission," and the form is still editable.
- Don’t forget to click th

In [27]:
response = openai_client.responses.create(
    model="gemma3:1b",
    input=prompt
)

In [28]:
response.output_text

"Okay, here are some summaries based on what’s provided:\n\n*   **Joining Now:** Yes, you can join regardless of registration, but homework submissions need deadlines! Don't procrastinate – submit them as soon as possible.\n*   **Homework After Scoring:** To submit homework, you *cannot* submit after the assignment is scored. It closes (scored) at that point.  Later submissions are rejected. Ensure you check your code against the ‘homework.md’ file if it’s been submitted late. \n*   **Zoomcamp Mode:**  You can join Zoomcamp without registering – use it for interest/data collection, even if there's no real-time group membership verification.\n*   **Certificate Collection:** You can only get a certificate *if you complete the course with a live cohort*. We don’t offer certificates for self-paced mode courses. \n\n*   **Starts With Reading & Finding Resources:** Get started by:\n    1.  Reading pins and bookmarks on the course channel.\n    2.  Viewing video lessons and reading repository

In [29]:
response.usage


ResponseUsage(input_tokens=794, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=350, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1144)

In [30]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]

response = openai_client.responses.create(
    model='gemma3:1b',
    input=message_history
)

In [31]:
response.output_text

'This is a great response! It efficiently covers all aspects of the question about joining a course after discovering it – offering helpful guidance to both those who are participating and those who might be hesitant. Here\'s a breakdown highlighting why it’s effective, broken down into strengths:\n\n**Strengths Summary:**\n\n* **Provides Context & Clarity:**  The responses effectively establish that while registration is possible, there are crucial deadlines for homework submission.\n*   **Clear Policy on Homework Late Submission:** The language precisely defines when late submissions are considered, alleviating potential confusion and frustration.\n* **Distinguishes Live/Self-Paced Modes:** This is key to understanding the different modes of operation – recognizing they operate differently concerning certificate granting (peer review). \n*  **Practical Guidance for Learners & Reviewers:** It gives sensible, actionable steps for both students and those helping others (e.g., searching,

In [32]:
def llm(instructions, user_prompt, model="gemma3:1b"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [35]:
def rag(query, model="gemma3:1b"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [36]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Okay, here’s a summary of the helpful responses regarding joining the course:

**General Questioning & Understanding:**

*   **Can I join the course after the start date?**:  Yes, you're still eligible to submit homework even if you haven’t registered. The form is still open for submissions.
* **Homework Submission Policy**: - Late assignments are not permitted beyond scoring; only the form is editable. It’s recommended you look at `homework.md` file before submission.
*   **Registration & Zoomcamp Status**: You *don't* need to register for a ZoomCamp track, it’s mainly for gauging interest and tracking data. It doesn’t prevent sending homework after the course opens.

**Scenario-Dependent Responses:**

*   **Self-Paced vs. Live Cohort**: Both modes are fine – you can join without registering, though registration is not tracked against registered users. The self-paced mode lacks certificate issuance for completion.
*  **(Certificate):** Can only be acquired by completing the course wit